# Imports

In [1]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [2]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/17 22:22:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Data

In [3]:
int_customers = spark.read.parquet("../data/02_intermediate/int_customers")
int_vehicles = spark.read.parquet("../data/02_intermediate/int_vehicles")

In [4]:
set(int_customers.columns).intersection(set(int_vehicles.columns))

{'creation_on', 'customer_id', 'modified_on', 'station_brand'}

In [5]:
df = int_customers.drop('creation_on', 'modified_on', 'station_brand').join(
    int_vehicles.drop('creation_on', 'modified_on', 'station_brand'),
    on="customer_id",
    how="inner"
)

# Create ID list

## Identify duplicates

In [6]:
df.count(), df.dropDuplicates().count()

25/05/17 22:22:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/05/17 22:22:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


(7074963, 7074963)

In [ ]:
df.filter(
    f.length("plate_number") == 7
).filter(
    f.length("mobile") == 10
).withColumn(
    "mobile",
    f.overlay(f.col("mobile"), f.lit("966"), 0, 1)
).withColumn(
    "_key",
    f.concat_ws("__", "customer_id", "customer_vehicle_id")
).withColumn(
    "_id",
    f.concat_ws("__", "plate_number", "mobile")
).select(
    f.countDistinct("_id"),
    f.countDistinct("_key"),
).show()

+-------------------+--------------------+
|count(DISTINCT _id)|count(DISTINCT _key)|
+-------------------+--------------------+
|            6066770|             6589967|
+-------------------+--------------------+



In [23]:
df.withColumn(
    "mobile",
    f.overlay(f.col("mobile"), f.lit("966"), 0, 2)
).show(50, truncate=False)

+------------------------------------+----------+---------------------------+------------+-------+-----+------+-----------+----------+-------------+--------+----------------+-------+----+---------+------------------------------------+----------+--------------+----------+-----------------+--------+------------+-----------------+---------------+--------+-------------------+
|customer_id                         |user_title|customer_name              |mobile      |mobile2|email|gender|nationality|birth_date|location_name|is_owner|is_cash_customer|payterm|age |is_active|customer_vehicle_id                 |maker     |model         |model_year|transmission_type|pms_name|plate_number|code_vin         |is_driver_owner|is_truck|vehicle_brand_level|
+------------------------------------+----------+---------------------------+------------+-------+-----+------+-----------+----------+-------------+--------+----------------+-------+----+---------+------------------------------------+----------+-----

In [13]:
from random import choice
from string import ascii_lowercase
n = 10

string_val = "".join(choice(ascii_lowercase) for i in range(n))
string_val

'pizbfxkgxi'

In [20]:
wrong_mobiles = [f"0{number1}" + f"{number2}"*8 for number1 in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] for number2 in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]
wrong_mobiles

['0000000000',
 '0011111111',
 '0022222222',
 '0033333333',
 '0044444444',
 '0055555555',
 '0066666666',
 '0077777777',
 '0088888888',
 '0099999999',
 '0100000000',
 '0111111111',
 '0122222222',
 '0133333333',
 '0144444444',
 '0155555555',
 '0166666666',
 '0177777777',
 '0188888888',
 '0199999999',
 '0200000000',
 '0211111111',
 '0222222222',
 '0233333333',
 '0244444444',
 '0255555555',
 '0266666666',
 '0277777777',
 '0288888888',
 '0299999999',
 '0300000000',
 '0311111111',
 '0322222222',
 '0333333333',
 '0344444444',
 '0355555555',
 '0366666666',
 '0377777777',
 '0388888888',
 '0399999999',
 '0400000000',
 '0411111111',
 '0422222222',
 '0433333333',
 '0444444444',
 '0455555555',
 '0466666666',
 '0477777777',
 '0488888888',
 '0499999999',
 '0500000000',
 '0511111111',
 '0522222222',
 '0533333333',
 '0544444444',
 '0555555555',
 '0566666666',
 '0577777777',
 '0588888888',
 '0599999999',
 '0600000000',
 '0611111111',
 '0622222222',
 '0633333333',
 '0644444444',
 '0655555555',
 '06666666

In [8]:
df.withColumn(
    "len_plate",
    f.length("plate_number")
).groupBy("len_plate").agg(
    f.count("customer_vehicle_id").alias("count")
).orderBy(f.desc("count")).show(20)

+---------+-------+
|len_plate|  count|
+---------+-------+
|        7|6855208|
|        6| 120199|
|        5|  44514|
|        4|  18872|
|        8|  17883|
|       10|   5587|
|        9|   4223|
|       11|   2423|
|       17|   1572|
|        3|   1271|
|       12|    845|
|       13|    804|
|       18|    438|
|        2|    337|
|       16|    318|
|        1|    194|
|       14|     87|
|        0|     80|
|       15|     59|
|       20|     37|
+---------+-------+
only showing top 20 rows



In [9]:
df.withColumn(
    "len_mobile",
    f.length("mobile")
).groupBy("len_mobile").agg(
    f.count("customer_vehicle_id").alias("count")
).orderBy(f.desc("count")).show(20)

+----------+-------+
|len_mobile|  count|
+----------+-------+
|        10|6798243|
|         1| 276720|
+----------+-------+



In [10]:
df.withColumn(
    "len_mobile",
    f.substr(f.col("mobile"), f.lit(0), f.lit(1))
).groupBy("len_mobile").agg(
    f.count("customer_vehicle_id").alias("count")
).orderBy(f.desc("count")).show(20)

+----------+-------+
|len_mobile|  count|
+----------+-------+
|         0|7074963|
+----------+-------+



## Create new hash

## Identify problematics